# Wu et al. (IEEE TCCN 2025) — Baseline Simulation
### "Joint AAV Deployment and Edge Association for Energy-Efficient Federated Learning"
**Architecture Mapping:**
| Wu et al. | Your System |
|---|---|
| UEs (terrestrial devices N) | Cluster Heads (CHs) |
| AAVs (aerial parameter servers M) | UAVs |
| Remote Cloud | Base Station |
| Local dataset D_n | CH aggregated data Q_h |
| Edge aggregation on AAV | UAV local FL training |
| Cloud aggregation | Base Station FedAvg |

**Algorithm:** FBGA — FL Bundle-based Greedy Association (Algorithm 5, Wu et al. TCCN 2025)
- For each candidate UAV location, greedily pick CHs with smallest communication latency t_com
- Select UAV location with best energy-effectiveness (min avg energy per served CH)
- Repeat until all CHs are served or budget exhausted
- Evaluated on K = Σ G_u (same reward metric as proposed)

**All physics parameters identical to proposed_fixed.ipynb (Singh et al. IEEE TSC 2024)**

## Cell 1 — Imports & Parameters

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
import random, warnings
warnings.filterwarnings('ignore')

# ============================================================
# PHYSICS PARAMETERS — identical to proposed_fixed.ipynb
# ============================================================
AREA_SIZE    = 200
NUM_SENSORS  = 2000
K            = 70
num_uavs     = 15

# Sensor/radio energy (Heinzelman 2002)
E_cir   = 50e-9
theta   = 10e-12
phi     = 0.0013e-12
D       = np.sqrt(theta / phi)
B       = 20e6
p_h     = 0.2
sigma   = 1e-9
gamma   = 1
H_alt   = 100
g_hu    = 1

# UAV propulsion (Zeng et al. IEEE TWC 2019)
C_PD    = 9.26e-4
C_RD    = 2250.0
V_u     = 10.0
v_amp   = 2.0
p_cir   = 0.1
E_MAX   = 10800.0

# FL computation/communication
mu_hu     = 1.0
W_model   = 1e6
f_hu      = 2e9
epsilon_u = 1e-26
alpha_u   = 0.5
beta_u    = 0.5
S_grad    = 1e6
b_u       = 20e6
SNR_u     = 20.0
P_com     = 0.1

# FL training
NUM_ROUNDS   = 5
LOCAL_EPOCHS = 3
LOCAL_BATCH  = 32
LR           = 1e-3
TEST_SIZE    = 0.2

# Reward (Singh et al. IEEE TSC 2024)
g_u_reward = 1.0
pi_u       = 1e-3

# ============================================================
# WU et al. TCCN 2025 — FBGA Parameters (Section V)
# ============================================================
# R_max: maximum coverage range of each AAV/UAV
# Wu (Section V-A): R_max = 100m in their evaluation
# In our 200x200m farm, UAV coverage radius = 150m (covers more CHs)
R_MAX_UAV   = 150.0    # UAV max coverage radius (m)

# UAV hovering power P_m and hover cost E_aav
# Wu (Section V-A): hovering time=5min, P_m=1W, E_aav=100J
P_HOVER     = 1.0      # W (hover power)
T_HOVER     = 300.0    # s (5 minutes hover time)
E_AAV       = 100.0    # J (edge aggregation energy per UAV)

# Wu FL parameters: v=10, δ=10, ε=0.1 (Section V-A)
V_WU        = 10.0     # local accuracy iteration constant
DELTA_WU    = 10.0     # edge convergence constant
EPS_EDGE    = 0.1      # edge accuracy ε
# I_eps_theta: number of edge iterations = δ·log(1/ε)/(1−θ)
THETA_LOCAL = 0.5      # local accuracy θ (fixed for simplicity)
I_EPS_THETA = DELTA_WU * np.log(1/EPS_EDGE) / (1 - THETA_LOCAL)

# Wu bandwidth B_m = 2 MHz per UAV (Section V-A)
B_WU        = 2e6      # Hz
# Wu channel: h0=-10dB, N0=-90dBm (Section V-A)
H0_WU       = 10**(-10/10)    # reference channel gain
N0_WU       = 10**(-90/10)*1e-3  # W
# Wu UE transmit power p_n = 1W (Section V-A)
P_UE        = 1.0      # W

print("=" * 60)
print("  WU et al. TCCN 2025 — FBGA BASELINE")
print(f"  Farm: {AREA_SIZE}x{AREA_SIZE}m | CHs: {K} | UAVs: {num_uavs}")
print(f"  R_MAX_UAV: {R_MAX_UAV}m | I_eps_theta: {I_EPS_THETA:.2f}")
print("=" * 60)

np.random.seed(42); random.seed(42)


## Cell 2 — Load Dataset

In [ ]:
try:
    df_full = pd.read_csv("agriculture_dataset.csv")
    print(f"Dataset loaded: {df_full.shape[0]:,} rows x {df_full.shape[1]} cols")
    USING_REAL = True
except FileNotFoundError:
    print("agriculture_dataset.csv not found — using synthetic data (3000 rows)")
    USING_REAL = False
    N = 3000
    df_full = pd.DataFrame({
        'High_Resolution_RGB'   : np.random.randint(0,2,N),
        'Multispectral_Images'  : np.random.randint(0,2,N),
        'Thermal_Images'        : np.random.randint(0,2,N),
        'Temporal_Images'       : np.random.randint(0,2,N),
        'Spatial_Resolution'    : np.random.uniform(0.01,2.7,N),
        'GPS_Coordinates'       : np.random.randint(100000,999999,N),
        'Field_Boundaries'      : np.random.randint(1,4,N),
        'Elevation_Data'        : np.random.uniform(10,100,N),
        'Canopy_Coverage'       : np.random.uniform(0,300,N),
        'NDVI'                  : np.random.uniform(0.05,1.05,N),
        'SAVI'                  : np.random.uniform(0.05,0.75,N),
        'Chlorophyll_Content'   : np.random.uniform(0.05,4.2,N),
        'Leaf_Area_Index'       : np.random.uniform(0.05,4.8,N),
        'Crop_Stress_Indicator' : np.random.randint(0,100,N),
        'Temperature'           : np.random.uniform(10,40,N),
        'Humidity'              : np.random.uniform(20,95,N),
        'Rainfall'              : np.random.uniform(0,130,N),
        'Wind_Speed'            : np.random.uniform(0.03,7.5,N),
        'Soil_Moisture'         : np.random.uniform(2,38,N),
        'Soil_pH'               : np.random.uniform(4.5,8.0,N),
        'Organic_Matter'        : np.random.uniform(0.001,20,N),
        'Pest_Hotspots'         : np.random.randint(0,2,N),
        'Weed_Coverage'         : np.random.uniform(0,9,N),
        'Pest_Damage'           : np.random.randint(0,100,N),
        'Crop_Growth_Stage'     : np.random.randint(1,5,N),
        'Expected_Yield'        : np.random.uniform(900,5200,N),
        'Crop_Type'             : np.random.choice(['Wheat','Maize','Rice','Soybean'],N),
        'Ground_Truth_Segmentation': np.random.randint(0,2,N),
        'Bounding_Boxes'        : np.random.randint(0,10,N),
        'Water_Flow'            : np.random.uniform(0,50,N),
        'Drainage_Features'     : np.random.randint(0,2,N),
    })
    lbl = np.zeros(N, dtype=int)
    for i in range(N):
        s = 0
        if 18 <= df_full.loc[i,'Temperature'] <= 34: s += 1
        if 5  <= df_full.loc[i,'Soil_pH'] <= 7.5:    s += 1
        if df_full.loc[i,'Soil_Moisture'] >= 15:      s += 1
        if df_full.loc[i,'NDVI'] >= 0.4:              s += 1
        if df_full.loc[i,'Crop_Stress_Indicator'] < 50: s += 1
        lbl[i] = 1 if s >= 3 else 0
    df_full['Crop_Health_Label'] = lbl

target_col   = 'Crop_Health_Label'
drop_cols    = ['Crop_Type','Ground_Truth_Segmentation','Bounding_Boxes']
feature_cols = [c for c in df_full.columns if c not in drop_cols + [target_col]]
df_full[feature_cols] = df_full[feature_cols].apply(pd.to_numeric, errors='coerce')
df_full[target_col]   = pd.to_numeric(df_full[target_col], errors='coerce')
imputer  = SimpleImputer(strategy='mean')
X_all_raw = imputer.fit_transform(df_full[feature_cols])
y_all    = df_full[target_col].fillna(0).values.astype(int)
scaler   = StandardScaler()
X_all    = scaler.fit_transform(X_all_raw)
N_FEATURES = X_all.shape[1]
print(f"Features: {N_FEATURES} | Samples: {len(y_all)} | "
      f"Healthy: {np.sum(y_all==1)} | Unhealthy: {np.sum(y_all==0)}")


## Cell 3 — Sensor Deployment + K-Means++ → Cluster Heads

In [ ]:
sensor_data = []
for i in range(NUM_SENSORS):
    x = np.random.uniform(0, AREA_SIZE)
    y = np.random.uniform(0, AREA_SIZE)
    sensor_data.append([f"SENS{str(i+1).zfill(4)}", x, y])

df = pd.DataFrame(sensor_data, columns=["sensor_id","x","y"])
df.set_index("sensor_id", inplace=True)
df["energy"] = np.random.uniform(2.0, 4.0, NUM_SENSORS)
df["k_n"]    = np.random.randint(2000, 6000, NUM_SENSORS)
df["I_np"]   = np.random.randint(0, 2, NUM_SENSORS)

kmeans = KMeans(n_clusters=K, init="k-means++", random_state=42, n_init=10)
df["cluster_id"] = kmeans.fit_predict(df[["x","y"]]).astype(int)

CH_list = []
for c in range(K):
    pts  = df[df["cluster_id"]==c]
    cen  = kmeans.cluster_centers_[c]
    dists = np.sqrt((pts["x"]-cen[0])**2 + (pts["y"]-cen[1])**2)
    CH_list.append(dists.idxmin())

stable_dict = {ch: [] for ch in CH_list}
for s_id, row in df.iterrows():
    stable_dict[CH_list[int(row["cluster_id"])]].append(s_id)

Q_h = {ch: sum(df.loc[s,"k_n"] for s in sensors)
       for ch, sensors in stable_dict.items()}

rho_h = {}
for h, ch_id in enumerate(CH_list):
    members = stable_dict[ch_id]
    rho_h[ch_id] = sum(df.loc[s,"I_np"] for s in members)

UAV_positions = [(np.random.uniform(0,AREA_SIZE),
                  np.random.uniform(0,AREA_SIZE)) for _ in range(num_uavs)]
UAV_BASE = np.array([AREA_SIZE/2, AREA_SIZE/2])

cluster_sizes = [len(stable_dict[ch]) for ch in CH_list]
print(f"Sensors: {NUM_SENSORS} | CHs: {K} | UAVs: {num_uavs}")
print(f"Avg cluster size: {np.mean(cluster_sizes):.1f}")

ch_table = pd.DataFrame({
    'CH'        : [f'CH{i+1:02d}' for i in range(K)],
    'Sensors'   : cluster_sizes,
    'Q_h(Mbits)': [round(Q_h[ch]/1e6, 2) for ch in CH_list],
    'rho_h'     : [round(rho_h[ch], 1) for ch in CH_list]
})
print("\n── CH Summary (first 10) ──")
print(ch_table.head(10).to_string(index=False))


## Cell 4 — Energy Functions (same physics as proposed)

In [ ]:
def propulsion_power(v_speed):
    return C_PD * v_speed**3 + C_RD / v_speed

P_u = propulsion_power(V_u)

def calc_E_sensor_tx(k_n, d):
    if d < D:
        return k_n * (theta * d**2 + E_cir)
    else:
        return k_n * (phi * d**4 + E_cir)

def calc_E_sensor_rx(C_nh, k_n):
    return C_nh * k_n * E_cir

def data_rate_R(ch_pos, uav_pos):
    dx  = ch_pos[0] - uav_pos[0]; dy = ch_pos[1] - uav_pos[1]
    d_3d = np.sqrt(dx**2 + dy**2 + H_alt**2)
    h    = g_hu / (d_3d**2)
    snr  = (p_h * h) / sigma
    return B * np.log2(1 + snr)

def calc_E_CH(Q_bits, R_bps):
    T = Q_bits / max(R_bps, 1e-9)
    return p_h * T

def calc_E_tve(route_length):
    return P_u * (route_length / V_u)

def calc_E_cmp_ch(Q_bits):
    return epsilon_u * (f_hu**2) * Q_bits * alpha_u * W_model

def calc_E_com():
    snr_lin = 10 ** (SNR_u / 10)
    R_up    = b_u * np.log2(1 + snr_lin)
    S_comp  = S_grad * alpha_u * beta_u
    T_com   = S_comp / R_up
    return T_com * P_com

def trans_time_T(Q_bits, R_bps):
    return Q_bits / max(R_bps, 1e-9)

# ── Wu et al. communication latency t_com (Eq.7 in Wu) ──────
# t_com_{mn} = s_n / r_{mn}  where r_{mn} uses Wu channel model
# In our system: s_n = Q_h[h] (data size), r uses 5G model
def wu_comm_latency(Q_bits, ch_pos, uav_pos):
    """
    Wu Eq.6-7: t_com_mn = s_n / r_mn
    r_mn = B * log2(1 + h0*p_n / (d_mn^2 + H^2) / N0)
    Mapped: s_n=Q_h[h], p_n=P_UE, B=B_WU, h0=H0_WU
    """
    dx   = ch_pos[0] - uav_pos[0]; dy = ch_pos[1] - uav_pos[1]
    d_mn = np.sqrt(dx**2 + dy**2)
    denom = (d_mn**2 + H_alt**2) * N0_WU
    snr  = (H0_WU * P_UE) / max(denom, 1e-30)
    r_mn = B_WU * np.log2(1 + snr)
    return Q_bits / max(r_mn, 1.0)

# ── Wu AAV hovering cost E_aav_m (Eq.9 in Wu) ───────────────
def wu_E_aav():
    """
    Wu Eq.9: E_aav = E_hover + E_ag_br
    E_hover = P_m * T  (hovering energy)
    E_ag_br = edge aggregation energy (constant = E_AAV)
    """
    E_hover = P_HOVER * T_HOVER
    return E_hover + E_AAV

E_AAV_M = wu_E_aav()

print(f"All energy functions ready.")
print(f"  P_u (propulsion) = {P_u:.1f} W  at v={V_u} m/s")
print(f"  E_AAV_M (Wu Eq.9) = {E_AAV_M:.1f} J per UAV")
print(f"  I_eps_theta = {I_EPS_THETA:.2f} edge iterations")


## Cell 5 — Wu Algorithm 5: FBGA — FL Bundle-based Greedy Association

**Algorithm 5 (Wu et al.) — Mapped to our system:**
1. Generate candidate UAV locations M_c from CH positions (FL bundles)
2. While unassigned CHs exist:
   a. For each candidate UAV location m:
      - Greedily pick unassigned CHs by smallest t_com_mn
      - Check latency feasibility (P3 solvability simplified to budget check)
      - Compute energy-effectiveness = E_edge_m / |N_m|
   b. Select m* = argmin(energy-effectiveness)
   c. Deploy UAV at m*, assign its CHs
3. Return UAV deployment Z and CH association A

In [ ]:
def fbga_assign(K_local, num_uavs_local, CH_list_local, Q_h_local,
                CH_px, CH_py, UAV_pos_local, seed=42):
    """
    Implements Wu et al. TCCN 2025, Algorithm 5 (FBGA)
    mapped to CH→UAV assignment.

    In our system:
    - CH positions serve as FL bundle anchor candidates
    - num_uavs_local UAVs are deployed greedily
    - Energy-effectiveness = E_edge_m / |N_m|

    Returns:
      M_h   : list[int|None] — UAV index per CH
      M_u   : list[list[int]] — CH indices per UAV
      E_rem : list[float] — remaining energy per UAV
    """
    rng = np.random.RandomState(seed)

    # ── Step 1: Candidate locations M_c = UAV positions (given) ──────────
    # In our system, UAV positions are pre-specified (like FL bundle anchors)
    # We treat each UAV as one candidate location (already deployed)
    # This is the direct analog of Wu's anchor point set M_c

    M_h   = [None] * K_local
    M_u   = [[] for _ in range(num_uavs_local)]
    E_rem = [E_MAX] * num_uavs_local

    unassigned_CHs = list(range(K_local))   # set N \ N' in Wu
    deployed_UAVs  = []                      # set M in Wu

    # ── Step 2: FBGA greedy loop (Algorithm 5, lines 3-15) ───────────────
    while unassigned_CHs:
        best_u         = None
        best_eff       = np.inf   # best energy-effectiveness (min)
        best_subset    = []

        # For each UAV not yet deployed (candidate location)
        for u in range(num_uavs_local):
            if u in deployed_UAVs:
                continue

            uav_pos = UAV_pos_local[u]

            # ── Algorithm 5 lines 7-12: greedily pick CHs by min t_com ──
            # Sort unassigned CHs by communication latency t_com_mn
            latencies = []
            for h in unassigned_CHs:
                ch_pos = (CH_px[h], CH_py[h])
                t_com  = wu_comm_latency(Q_h_local[h], ch_pos, uav_pos)
                latencies.append((t_com, h))
            latencies.sort(key=lambda x: x[0])

            # Greedily add CHs while E_MAX budget holds
            e_tve_running = 0.0
            e_cmp_running = 0.0
            e_ch_tx_running = 0.0
            N_m = []
            flag = True

            for t_com, h in latencies:
                ch_pos = (CH_px[h], CH_py[h])
                dist   = np.sqrt((CH_px[h]-uav_pos[0])**2 + (CH_py[h]-uav_pos[1])**2)
                e_tve_h   = calc_E_tve(dist)
                e_cmp_h   = calc_E_cmp_ch(Q_h_local[h])
                R         = data_rate_R(ch_pos, uav_pos)
                e_ch_tx_h = calc_E_CH(Q_h_local[h], R)

                # Feasibility check: total traversal + compute must fit E_MAX
                candidate_tve = e_tve_running + e_tve_h
                candidate_cmp = e_cmp_running + e_cmp_h

                if candidate_tve <= E_MAX and (e_tve_h + e_cmp_h) > 0:
                    N_m.append(h)
                    e_tve_running   += e_tve_h
                    e_cmp_running   += e_cmp_h
                    e_ch_tx_running += e_ch_tx_h
                else:
                    # Budget exceeded — stop adding for this UAV
                    flag = False
                    break

            if not N_m:
                continue

            # ── Energy-effectiveness (Wu Eq.20): E_edge_m / |N_m| ────────
            # E_edge_m = Σ I_eps_theta*(e_cmp+e_com_ch) + E_aav_m
            e_com_u = calc_E_com()
            E_edge_m = (I_EPS_THETA * (e_cmp_running + e_ch_tx_running) +
                        E_AAV_M + e_tve_running)
            effectiveness = E_edge_m / len(N_m)

            if effectiveness < best_eff:
                best_eff    = effectiveness
                best_u      = u
                best_subset = N_m.copy()

        if best_u is None:
            # No feasible UAV can serve remaining CHs
            break

        # ── Algorithm 5 lines 14-15: Deploy best UAV ─────────────────────
        deployed_UAVs.append(best_u)
        for h in best_subset:
            M_h[h]        = best_u
            M_u[best_u].append(h)
            ch_pos        = (CH_px[h], CH_py[h])
            dist          = np.sqrt((CH_px[h]-UAV_pos_local[best_u][0])**2 +
                                    (CH_py[h]-UAV_pos_local[best_u][1])**2)
            E_rem[best_u] -= calc_E_tve(dist)
            unassigned_CHs.remove(h)

    matched_count = sum(1 for x in M_h if x is not None)
    print(f"FBGA deployed UAVs: {len(deployed_UAVs)}")
    print(f"CHs assigned: {matched_count}/{K_local}")
    for u in deployed_UAVs:
        print(f"  UAV{u+1:02d}: {len(M_u[u])} CHs | E_rem={E_rem[u]:.1f}J")

    return M_h, M_u, E_rem


# Build CH position arrays for the main simulation
CH_pos_x_main = np.array([df.loc[CH_list[h],"x"] for h in range(K)])
CH_pos_y_main = np.array([df.loc[CH_list[h],"y"] for h in range(K)])
Q_h_arr_main  = np.array([Q_h[CH_list[h]] for h in range(K)])

# Run FBGA
M_h, M_u, E_rem = fbga_assign(
    K, num_uavs, CH_list, Q_h_arr_main,
    CH_pos_x_main, CH_pos_y_main, UAV_positions, seed=42)


## Cell 6 — FL Training (FedAvg)

In [ ]:
from sklearn.linear_model import SGDClassifier

def local_train(X, y, global_coef, global_intercept, epochs=LOCAL_EPOCHS):
    if len(X) == 0 or len(np.unique(y)) < 2:
        return global_coef.copy(), global_intercept.copy()
    model = SGDClassifier(loss='log_loss', max_iter=epochs, learning_rate='constant',
                          eta0=LR, random_state=42)
    model.coef_      = global_coef.copy()
    model.intercept_ = global_intercept.copy()
    model.classes_   = np.array([0, 1])
    for _ in range(epochs):
        idx = np.random.permutation(len(X))
        for start in range(0, len(X), LOCAL_BATCH):
            batch = idx[start:start+LOCAL_BATCH]
            if len(batch) < 2: continue
            model.partial_fit(X[batch], y[batch], classes=[0,1])
    return model.coef_.copy(), model.intercept_.copy()

def fedavg(updates, weights):
    total_w = sum(weights)
    if total_w == 0:
        return updates[0][0], updates[0][1]
    agg_coef = sum(w * c for (c,_),w in zip(updates,weights)) / total_w
    agg_int  = sum(w * i for (_,i),w in zip(updates,weights)) / total_w
    return agg_coef, agg_int

rows_per_cluster = max(1, len(df_full) // K)
shuffled_idx     = np.random.permutation(len(df_full))
cluster_row_map  = {}
for h in range(K):
    s = h * rows_per_cluster
    e = s + rows_per_cluster if h < K-1 else len(df_full)
    cluster_row_map[h] = shuffled_idx[s:e]

uav_data   = {}
uav_travel = {}
for u in range(num_uavs):
    matched = M_u[u]
    if not matched:
        uav_data[u]   = (np.zeros((0,N_FEATURES)), np.zeros(0,dtype=int))
        uav_travel[u] = 0.0
        continue
    Xu, yu = [], []
    for h in matched:
        idx = cluster_row_map[h]
        Xu.append(X_all[idx]); yu.append(y_all[idx])
    uav_data[u] = (np.vstack(Xu), np.concatenate(yu))
    route = ([UAV_BASE] +
             [np.array([df.loc[CH_list[h],"x"], df.loc[CH_list[h],"y"]]) for h in matched] +
             [UAV_BASE])
    uav_travel[u] = sum(np.linalg.norm(route[i+1]-route[i]) for i in range(len(route)-1))

active_uavs = [u for u in range(num_uavs) if len(uav_data[u][1]) > 0]

g_coef = np.zeros((1, N_FEATURES))
g_int  = np.zeros(1)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=42, stratify=y_all)

fl_acc_rounds = []
print(f"Active UAVs: {len(active_uavs)} | Starting FedAvg...")

for rnd in range(NUM_ROUNDS):
    local_updates = []; local_weights = []
    for u in active_uavs:
        X_u, y_u = uav_data[u]
        if len(X_u) < 2: continue
        c, i = local_train(X_u, y_u, g_coef, g_int, LOCAL_EPOCHS)
        local_updates.append((c, i)); local_weights.append(len(y_u))
    if not local_updates: break
    g_coef, g_int = fedavg(local_updates, local_weights)
    ev = SGDClassifier(loss='log_loss')
    ev.coef_ = g_coef; ev.intercept_ = g_int; ev.classes_ = np.array([0,1])
    acc = accuracy_score(y_te, ev.predict(X_te))
    fl_acc_rounds.append(acc)
    print(f"  Round {rnd+1}/{NUM_ROUNDS} — Accuracy: {acc:.4f}")

final_acc = fl_acc_rounds[-1] if fl_acc_rounds else 0.0
print(f"\nFinal FL Accuracy: {final_acc:.4f}")


## Cell 7 — Compute Energy Metrics & K_reward

In [ ]:
E_sensor_tx = 0
for s_id, row in df.iterrows():
    ch_id = CH_list[int(row["cluster_id"])]
    d     = np.sqrt((row["x"]-df.loc[ch_id,"x"])**2 + (row["y"]-df.loc[ch_id,"y"])**2)
    E_sensor_tx += calc_E_sensor_tx(row["k_n"], d)

E_sensor_rx = sum(
    calc_E_sensor_rx(1, df.loc[s,"k_n"])
    for ch in CH_list for s in stable_dict[ch])

E_ch_tx = 0
T_trans = 0
for h in range(K):
    if M_h[h] is None: continue
    u      = M_h[h]
    ch_pos = (df.loc[CH_list[h],"x"], df.loc[CH_list[h],"y"])
    R      = data_rate_R(ch_pos, UAV_positions[u])
    E_ch_tx += calc_E_CH(Q_h[CH_list[h]], R)
    T_trans += trans_time_T(Q_h[CH_list[h]], R)

E_tve = sum(E_MAX - E_rem[u] for u in range(num_uavs))

E_cmp = sum(
    sum(calc_E_cmp_ch(Q_h[CH_list[h]]) for h in M_u[u])
    for u in range(num_uavs))

E_com = num_uavs * calc_E_com()

E_total = E_sensor_tx + E_sensor_rx + E_ch_tx + E_tve + E_cmp + E_com
matched_count = sum(1 for x in M_h if x is not None)

# K_reward (Singh et al. Eq.22-25)
K_reward = 0.0
for u in range(num_uavs):
    if not M_u[u]: continue
    E_tare_u = E_MAX - E_rem[u]
    E_cmp_u  = sum(calc_E_cmp_ch(Q_h[CH_list[h]]) for h in M_u[u])
    E_com_u  = calc_E_com()
    R_u      = g_u_reward * final_acc
    G_u      = R_u - pi_u * (E_tare_u + E_cmp_u + E_com_u)
    K_reward += G_u

total_cost = pi_u * E_total

print("── WU et al. TCCN 2025 — Main Simulation Energy ──")
print(f"  E_sensor_tx (Eq.2)  : {E_sensor_tx:.4e} J")
print(f"  E_sensor_rx (Eq.rec): {E_sensor_rx:.4e} J")
print(f"  E_CH_tx     (Eq.11) : {E_ch_tx:.4e} J")
print(f"  E_UAV_tve   (Eq.12) : {E_tve:.4e} J")
print(f"  E_UAV_cmp   (Eq.17) : {E_cmp:.4e} J")
print(f"  E_UAV_com   (Eq.20) : {E_com:.4e} J")
print(f"  E_total             : {E_total:.4e} J")
print(f"  T_trans             : {T_trans:.4e} s")
print(f"  Matched CHs         : {matched_count}/{K}")
print(f"  FL Accuracy         : {final_acc:.4f}")
print(f"  Total Cost          : {total_cost:.4e}")
print(f"  K_reward (Eq.25)    : {K_reward:.6f}")


## Cell 8 — Parametric Sweep: run_one function

In [ ]:
from sklearn.linear_model import SGDClassifier

def run_one_wu(n_sensors, n_uavs, K_ch=None, seed=42):
    """
    Run one simulation point with Wu et al. FBGA algorithm.
    """
    rng = np.random.RandomState(seed)

    if K_ch is None:
        K_ch = max(5, n_sensors // 30)

    # ── Sensor deployment ──────────────────────────────────────
    sx  = rng.uniform(0, AREA_SIZE, n_sensors)
    sy  = rng.uniform(0, AREA_SIZE, n_sensors)
    k_n = rng.randint(2000, 6000, n_sensors)

    # ── K-Means CH selection ───────────────────────────────────
    coords = np.column_stack([sx, sy])
    km = KMeans(n_clusters=K_ch, init='k-means++', random_state=seed, n_init=10)
    labels = km.fit_predict(coords)

    CH_px_l = np.zeros(K_ch); CH_py_l = np.zeros(K_ch)
    Q_h_l   = np.zeros(K_ch)
    for c in range(K_ch):
        mask = (labels == c)
        cen  = km.cluster_centers_[c]
        dists = np.sqrt((sx[mask]-cen[0])**2 + (sy[mask]-cen[1])**2)
        idx_ch = np.where(mask)[0][np.argmin(dists)]
        CH_px_l[c] = sx[idx_ch]; CH_py_l[c] = sy[idx_ch]
        Q_h_l[c]   = np.sum(k_n[mask])

    uav_px = rng.uniform(0, AREA_SIZE, n_uavs)
    uav_py = rng.uniform(0, AREA_SIZE, n_uavs)
    uav_positions_l = list(zip(uav_px, uav_py))

    # ── FBGA ──────────────────────────────────────────────────
    M_h_l  = [None] * K_ch
    M_u_l  = [[] for _ in range(n_uavs)]
    E_rem_l = [E_MAX] * n_uavs

    unassigned = list(range(K_ch))
    deployed   = []

    while unassigned:
        best_u = None; best_eff = np.inf; best_sub = []
        for u in range(n_uavs):
            if u in deployed: continue
            uav_pos = uav_positions_l[u]

            lat = [(wu_comm_latency(Q_h_l[h], (CH_px_l[h],CH_py_l[h]), uav_pos), h)
                   for h in unassigned]
            lat.sort()

            e_tve_r = 0.0; e_cmp_r = 0.0; e_chtx_r = 0.0
            N_m = []
            for t_com, h in lat:
                dist    = np.sqrt((CH_px_l[h]-uav_pos[0])**2 + (CH_py_l[h]-uav_pos[1])**2)
                e_tve_h = calc_E_tve(dist)
                e_cmp_h = epsilon_u * (f_hu**2) * Q_h_l[h] * alpha_u * W_model
                d_3d    = np.sqrt(dist**2 + H_alt**2)
                h_link  = g_hu / (d_3d**2)
                snr_    = (p_h * h_link) / sigma
                R_      = B * np.log2(1 + snr_)
                e_chtx_h = p_h * (Q_h_l[h] / max(R_, 1e-9))

                if e_tve_r + e_tve_h <= E_MAX:
                    N_m.append(h)
                    e_tve_r += e_tve_h; e_cmp_r += e_cmp_h; e_chtx_r += e_chtx_h
                else:
                    break

            if not N_m: continue

            E_edge_m = I_EPS_THETA * (e_cmp_r + e_chtx_r) + E_AAV_M + e_tve_r
            eff = E_edge_m / len(N_m)
            if eff < best_eff:
                best_eff = eff; best_u = u; best_sub = N_m[:]

        if best_u is None: break

        deployed.append(best_u)
        for h in best_sub:
            M_h_l[h] = best_u; M_u_l[best_u].append(h)
            dist = np.sqrt((CH_px_l[h]-uav_positions_l[best_u][0])**2 +
                           (CH_py_l[h]-uav_positions_l[best_u][1])**2)
            E_rem_l[best_u] -= calc_E_tve(dist)
            unassigned.remove(h)

    matched_count = sum(1 for x in M_h_l if x is not None)

    # ── Sensor→CH energies ─────────────────────────────────────
    E_tx_total = 0; E_rx_total = 0
    for i in range(n_sensors):
        c = labels[i]
        d = np.sqrt((sx[i]-CH_px_l[c])**2 + (sy[i]-CH_py_l[c])**2)
        E_tx_total += calc_E_sensor_tx(k_n[i], d)
        E_rx_total += calc_E_sensor_rx(1, k_n[i])

    # ── CH→UAV tx energy & T_trans ────────────────────────────
    E_ch_tx_l = 0; T_trans_l = 0
    for h in range(K_ch):
        if M_h_l[h] is None: continue
        u   = M_h_l[h]
        dx  = CH_px_l[h]-uav_px[u]; dy = CH_py_l[h]-uav_py[u]
        d3d = np.sqrt(dx**2+dy**2+H_alt**2)
        h_l = g_hu/(d3d**2)
        snr = (p_h*h_l)/sigma
        R   = B*np.log2(1+snr)
        T   = Q_h_l[h]/max(R,1e-9)
        E_ch_tx_l += p_h*T; T_trans_l += T

    E_tve_l = sum(E_MAX-E_rem_l[u] for u in range(n_uavs))
    E_cmp_l = sum(epsilon_u*(f_hu**2)*Q_h_l[h]*alpha_u*W_model
                  for u in range(n_uavs) for h in M_u_l[u])

    snr_lin = 10**(SNR_u/10)
    R_up    = b_u*np.log2(1+snr_lin)
    T_com   = (S_grad*alpha_u*beta_u)/R_up
    E_com_l = n_uavs*T_com*P_com

    E_total_l = E_tx_total+E_rx_total+E_ch_tx_l+E_tve_l+E_cmp_l+E_com_l

    # ── FL simulation (coverage-based) ─────────────────────────
    coverage = matched_count / max(K_ch, 1)
    n_samples = max(50, int(len(X_all)*coverage))
    idx_s = rng.choice(len(X_all), size=min(n_samples,len(X_all)), replace=False)
    X_sub = X_all[idx_s]; y_sub = y_all[idx_s]
    if len(np.unique(y_sub)) >= 2:
        X_tr2,X_te2,y_tr2,y_te2 = train_test_split(
            X_sub,y_sub,test_size=TEST_SIZE,random_state=seed)
        mfl = SGDClassifier(loss='log_loss',max_iter=NUM_ROUNDS*LOCAL_EPOCHS,
                            learning_rate='constant',eta0=LR,random_state=seed)
        if len(np.unique(y_tr2)) >= 2:
            mfl.fit(X_tr2,y_tr2)
            acc = accuracy_score(y_te2,mfl.predict(X_te2))
        else:
            acc = float(np.mean(y_te2==y_tr2[0]))
    else:
        acc = 0.5

    # ── K_reward ──────────────────────────────────────────────
    K_reward_l = 0.0
    for u in range(n_uavs):
        if not M_u_l[u]: continue
        E_tare_u = E_MAX-E_rem_l[u]
        E_cmp_u  = sum(epsilon_u*(f_hu**2)*Q_h_l[h]*alpha_u*W_model for h in M_u_l[u])
        E_com_u  = T_com*P_com
        G_u      = g_u_reward*acc - pi_u*(E_tare_u+E_cmp_u+E_com_u)
        K_reward_l += G_u

    return {
        'n_sensors'  : n_sensors,
        'K'          : K_ch,
        'n_uavs'     : n_uavs,
        'matched'    : matched_count,
        'E_tx'       : E_tx_total,
        'E_rx'       : E_rx_total,
        'T_trans'    : T_trans_l,
        'E_total'    : E_total_l,
        'total_cost' : pi_u*E_total_l,
        'K_reward'   : K_reward_l,
        'acc'        : acc,
    }

print("run_one_wu() defined.")


## Cell 9 — Case 1: Varying Sensors (UAVs=7)

In [ ]:
sensor_range_1 = [200, 400, 600, 800, 1000]
results1 = []
for ns in sensor_range_1:
    r = run_one_wu(ns, 7, seed=42)
    results1.append(r)
    print(f"  n_sensors={ns:5d} | K={r['K']:2d} | matched={r['matched']:2d} | "
          f"E_total={r['E_total']:.2e} | K_reward={r['K_reward']:.4f} | acc={r['acc']:.4f}")

df1 = pd.DataFrame(results1)
print("\nCase 1 complete.")
print(df1[['n_sensors','K','n_uavs','matched','E_tx','E_rx',
           'T_trans','E_total','total_cost','K_reward','acc']].to_string(index=False))


## Cell 10 — Case 2: Varying Sensors (UAVs=15)

In [ ]:
sensor_range_2 = list(range(200, 2001, 200))
results2 = []
for ns in sensor_range_2:
    r = run_one_wu(ns, 15, seed=42)
    results2.append(r)
    print(f"  n_sensors={ns:5d} | K={r['K']:2d} | matched={r['matched']:2d} | "
          f"E_total={r['E_total']:.2e} | K_reward={r['K_reward']:.4f} | acc={r['acc']:.4f}")

df2 = pd.DataFrame(results2)
print("\nCase 2 complete.")
print(df2[['n_sensors','K','n_uavs','matched','E_tx','E_rx',
           'T_trans','E_total','total_cost','K_reward','acc']].to_string(index=False))


## Cell 11 — Plots (matching proposed_fixed_with_acc.ipynb style)
### 10 metric plots (individual fig,ax per plot — identical layout to proposed)


In [ ]:
import matplotlib
matplotlib.use('Agg')

ALGO_LABEL  = 'Wu et al. (TCCN 2025)'
ALGO_COLOR  = 'steelblue'
ALGO_MARKER = '^'

# ── Plot 5 — Transmission Energy E_tx  (df2: 200→2000, UAVs=15) ─────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['E_tx'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='steelblue',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title(f'Transmission Energy E_tx — {ALGO_LABEL}', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors [UAVs=15, N=200→2000]', fontsize=11)
ax.set_ylabel('E_tx (J)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(x) for x in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'Sensor→CH transmission energy', transform=ax.transAxes,
        fontsize=8, ha='right', va='bottom', color='gray')
ax.legend([ALGO_LABEL], fontsize=9)
plt.tight_layout()
plt.savefig('wu_plot5_Etx.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: wu_plot5_Etx.png')

# ── Plot 6 — Reception Energy E_rx  (df2: 200→2000) ──────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
y_vals = df2['E_rx'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='darkorange',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title(f'Reception Energy E_rx — {ALGO_LABEL}', fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors [UAVs=15, N=200→2000]', fontsize=11)
ax.set_ylabel('E_rx (J)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(x) for x in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'CH reception energy', transform=ax.transAxes,
        fontsize=8, ha='right', va='bottom', color='gray')
ax.legend([ALGO_LABEL], fontsize=9)
plt.tight_layout()
plt.savefig('wu_plot6_Erx.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: wu_plot6_Erx.png')

# ── Plot 7 — Total System Energy E_total — Case 1  (UAVs=7) ──────────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals1 = df1['n_sensors'].tolist()
y_vals = df1['E_total'].tolist()
bars = ax.bar(range(len(x_vals1)), y_vals, color='crimson',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title(f'Total System Energy E_total — Case 1 (UAVs=7) — {ALGO_LABEL}',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors [Case 1: UAVs=7, N=200→1000]', fontsize=11)
ax.set_ylabel('E_total (J)', fontsize=11)
ax.set_xticks(range(len(x_vals1)))
ax.set_xticklabels([str(x) for x in x_vals1], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'E_tx+E_rx+E_CH→UAV+E_tve', transform=ax.transAxes,
        fontsize=8, ha='right', va='bottom', color='gray')
ax.legend([ALGO_LABEL], fontsize=9)
plt.tight_layout()
plt.savefig('wu_plot7_Etotal_case1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: wu_plot7_Etotal_case1.png')

# ── Plot 8 — Total System Energy E_total — Case 2  (UAVs=15) ─────────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals2 = df2['n_sensors'].tolist()
y_vals = df2['E_total'].tolist()
bars = ax.bar(range(len(x_vals2)), y_vals, color='firebrick',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title(f'Total System Energy E_total — Case 2 (UAVs=15) — {ALGO_LABEL}',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors [Case 2: UAVs=15, N=200→2000]', fontsize=11)
ax.set_ylabel('E_total (J)', fontsize=11)
ax.set_xticks(range(len(x_vals2)))
ax.set_xticklabels([str(x) for x in x_vals2], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'E_tx+E_rx+E_CH→UAV+E_tve', transform=ax.transAxes,
        fontsize=8, ha='right', va='bottom', color='gray')
ax.legend([ALGO_LABEL], fontsize=9)
plt.tight_layout()
plt.savefig('wu_plot8_Etotal_case2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: wu_plot8_Etotal_case2.png')

# ── Plot 9 — Total Transmission Time T_trans — Case 1 (UAVs=7) ───────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df1['K'].tolist()
y_vals = df1['T_trans'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='purple',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title(f'Total Transmission Time T_trans — Case 1 (UAVs=7) — {ALGO_LABEL}',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Cluster Heads K (dynamic)', fontsize=11)
ax.set_ylabel('T_trans (s)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(k) for k in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'More CHs → more total Q_h → higher T_trans',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
sensors_1 = df1['n_sensors'].tolist()
for ii, (k, s) in enumerate(zip(x_vals, sensors_1)):
    ax.text(ii, -ax.get_ylim()[1]*0.06, f'N={s}', ha='center', fontsize=7, color='dimgray')
ax.text(0.02, -0.08, 'Sensors:', transform=ax.transAxes, fontsize=7, color='dimgray')
ax.legend([ALGO_LABEL], fontsize=9)
plt.tight_layout()
plt.savefig('wu_plot9_Ttrans_case1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: wu_plot9_Ttrans_case1.png')

# ── Plot 10 — Total Transmission Time T_trans — Case 2 (UAVs=15) ─────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['K'].tolist()
y_vals = df2['T_trans'].tolist()
bars = ax.bar(range(len(x_vals)), y_vals, color='darkviolet',
              edgecolor='black', linewidth=0.8, alpha=0.88, width=0.55)
for bar, v in zip(bars, y_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
            f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title(f'Total Transmission Time T_trans — Case 2 (UAVs=15) — {ALGO_LABEL}',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Cluster Heads K (dynamic)', fontsize=11)
ax.set_ylabel('T_trans (s)', fontsize=11)
ax.set_xticks(range(len(x_vals)))
ax.set_xticklabels([str(k) for k in x_vals], fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.text(0.98, 0.02, 'More CHs → more total Q_h → higher T_trans',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
sensors_2 = df2['n_sensors'].tolist()
for ii, (k, s) in enumerate(zip(x_vals, sensors_2)):
    ax.text(ii, -ax.get_ylim()[1]*0.06, f'N={s}', ha='center', fontsize=7, color='dimgray')
ax.text(0.02, -0.08, 'Sensors:', transform=ax.transAxes, fontsize=7, color='dimgray')
ax.legend([ALGO_LABEL], fontsize=9)
plt.tight_layout()
plt.savefig('wu_plot10_Ttrans_case2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: wu_plot10_Ttrans_case2.png')

# ── Plot 11 — Total Cost — Case 1  (UAVs=7) ──────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df1['n_sensors'].tolist()
y_vals = df1['total_cost'].tolist()
ax.plot(x_vals, y_vals, marker=ALGO_MARKER, lw=3, ms=10,
        color='saddlebrown', label=ALGO_LABEL)
ax.fill_between(x_vals, y_vals, alpha=0.12, color='saddlebrown')
for x, y in zip(x_vals, y_vals):
    ax.annotate(f'{y:.4f}', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')
ax.set_title(f'Total Cost — Case 1 (UAVs=7) — {ALGO_LABEL}',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors [Case 1: UAVs=7, N=200→1000]', fontsize=11)
ax.set_ylabel('Total Cost', fontsize=11)
ax.set_xticks(x_vals)
ax.tick_params(axis='x', labelsize=9)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(0.98, 0.02, 'λ_e×E_total + λ_t×T_trans', transform=ax.transAxes,
        fontsize=8, ha='right', va='bottom', color='gray')
plt.tight_layout()
plt.savefig('wu_plot11_cost_case1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: wu_plot11_cost_case1.png')

# ── Plot 12 — Total Cost — Case 2  (UAVs=15) ─────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['total_cost'].tolist()
ax.plot(x_vals, y_vals, marker=ALGO_MARKER, lw=3, ms=10,
        color='chocolate', label=ALGO_LABEL)
ax.fill_between(x_vals, y_vals, alpha=0.12, color='chocolate')
for x, y in zip(x_vals, y_vals):
    ax.annotate(f'{y:.4f}', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')
ax.set_title(f'Total Cost — Case 2 (UAVs=15) — {ALGO_LABEL}',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors [Case 2: UAVs=15, N=200→2000]', fontsize=11)
ax.set_ylabel('Total Cost', fontsize=11)
ax.set_xticks(x_vals)
ax.tick_params(axis='x', labelsize=9)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(0.98, 0.02, 'λ_e×E_total + λ_t×T_trans', transform=ax.transAxes,
        fontsize=8, ha='right', va='bottom', color='gray')
plt.tight_layout()
plt.savefig('wu_plot12_cost_case2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: wu_plot12_cost_case2.png')

# ── Plot 13 — Total Reward K (Eq.25) — Case 1  (UAVs=7) ──────────────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df1['n_sensors'].tolist()
y_vals = df1['K_reward'].tolist()
ax.plot(x_vals, y_vals, marker=ALGO_MARKER, lw=3, ms=10,
        color='darkgreen', label=ALGO_LABEL)
ax.fill_between(x_vals, y_vals, alpha=0.12, color='darkgreen')
for x, y in zip(x_vals, y_vals):
    ax.annotate(f'{y:.4f}', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')
ax.set_title(f'Total Revenue K (Eq.25) — Case 1 (UAVs=7) — {ALGO_LABEL}',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors [Case 1: UAVs=7, N=200→1000]', fontsize=11)
ax.set_ylabel('K = Σ G_u', fontsize=11)
ax.set_xticks(x_vals)
ax.tick_params(axis='x', labelsize=9)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(0.98, 0.02, 'G_u = R_u − π_u(E_tare+E_cmp+E_com)',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
plt.tight_layout()
plt.savefig('wu_plot13_reward_case1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: wu_plot13_reward_case1.png')

# ── Plot 14 — Total Reward K (Eq.25) — Case 2  (UAVs=15) ─────────────────
fig, ax = plt.subplots(figsize=(10, 6))
x_vals = df2['n_sensors'].tolist()
y_vals = df2['K_reward'].tolist()
ax.plot(x_vals, y_vals, marker=ALGO_MARKER, lw=3, ms=10,
        color='seagreen', label=ALGO_LABEL)
ax.fill_between(x_vals, y_vals, alpha=0.12, color='seagreen')
for x, y in zip(x_vals, y_vals):
    ax.annotate(f'{y:.4f}', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')
ax.set_title(f'Total Revenue K (Eq.25) — Case 2 (UAVs=15) — {ALGO_LABEL}',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Number of Sensors [Case 2: UAVs=15, N=200→2000]', fontsize=11)
ax.set_ylabel('K = Σ G_u', fontsize=11)
ax.set_xticks(x_vals)
ax.tick_params(axis='x', labelsize=9)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(0.98, 0.02, 'G_u = R_u − π_u(E_tare+E_cmp+E_com)',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom', color='gray')
plt.tight_layout()
plt.savefig('wu_plot14_reward_case2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: wu_plot14_reward_case2.png')

# ── Plot 15 — FL Accuracy per Sensor Count — Case 1 & 2 ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
x_vals = df1['n_sensors'].tolist()
y_vals = df1['acc'].tolist()
ax.plot(x_vals, y_vals, marker=ALGO_MARKER, lw=3, ms=10,
        color='royalblue', label=f'{ALGO_LABEL} (UAVs=7)')
ax.fill_between(x_vals, y_vals, alpha=0.12, color='royalblue')
for x, y in zip(x_vals, y_vals):
    ax.annotate(f'{y:.4f}', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')
ax.set_title(f'FL Accuracy — Case 1 (UAVs=7)', fontweight='bold', fontsize=12)
ax.set_xlabel('Number of Sensors', fontsize=11); ax.set_ylabel('Accuracy', fontsize=11)
ax.set_ylim(0, 1.05); ax.set_xticks(x_vals); ax.tick_params(axis='x', labelsize=9)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
x_vals = df2['n_sensors'].tolist()
y_vals = df2['acc'].tolist()
ax.plot(x_vals, y_vals, marker=ALGO_MARKER, lw=3, ms=10,
        color='steelblue', label=f'{ALGO_LABEL} (UAVs=15)')
ax.fill_between(x_vals, y_vals, alpha=0.12, color='steelblue')
for x, y in zip(x_vals, y_vals):
    ax.annotate(f'{y:.4f}', (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9, fontweight='bold')
ax.set_title(f'FL Accuracy — Case 2 (UAVs=15)', fontweight='bold', fontsize=12)
ax.set_xlabel('Number of Sensors', fontsize=11); ax.set_ylabel('Accuracy', fontsize=11)
ax.set_ylim(0, 1.05); ax.set_xticks(x_vals); ax.tick_params(axis='x', labelsize=9)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

fig.suptitle(f'FL Training Accuracy — {ALGO_LABEL}', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('wu_plot15_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: wu_plot15_accuracy.png')


## Final Summary

In [ ]:
print("=" * 68)
print("  WU et al. (TCCN 2025) — FINAL SIMULATION SUMMARY")
print("=" * 68)
print(f"  Algorithm : FBGA — FL Bundle-based Greedy Association")
print(f"  Mapping   : CH=UE | UAV=AAV | BS=Remote Cloud")
print(f"  Farm: {AREA_SIZE}x{AREA_SIZE}m | FL rounds: {NUM_ROUNDS} | Epochs: {LOCAL_EPOCHS}")
print()
print("  Case 1 — UAVs=7, Sensors=200..1000")
print(df1[['n_sensors','K','n_uavs','matched','E_tx','E_rx',
           'T_trans','E_total','total_cost','K_reward','acc']].to_string(index=False))
print()
print("  Case 2 — UAVs=15, Sensors=200..2000 step 200")
print(df2[['n_sensors','K','n_uavs','matched','E_tx','E_rx',
           'T_trans','E_total','total_cost','K_reward','acc']].to_string(index=False))
print("=" * 68)
